# E3 — Error Analysis: LightGCN vs RLMRec-Con (SNA-MXH)

**Mục đích:** trả lời góp ý của thầy: chọn các user/item mà thêm ngữ nghĩa giúp hoặc làm giảm kết quả; đối chiếu profile ngữ nghĩa, lịch sử train và sách được gợi ý để giải thích, từ đó chọn cách điều chỉnh alignment cần kiểm chứng.  
**Input:** `trn_mat.pkl`, `tst_mat.pkl`, `usr_prf.pkl`, `itm_prf.pkl`, `usr_emb_np.pkl`, `itm_emb_np.pkl`, Top-K E1 của LightGCN và RLMRec-Con (3 seeds), `data/manifests/amazon_book_titles.json`.  
**Output:** `experiments/E3/error_analysis/` (per-user, nhóm user, phân tích theo mức, phía item, case study).  
**Không training / không inference.**

| Thành phần | Quy ước |
|---|---|
| ΔRecall@20 | Recall@20 RLMRec-Con − Recall@20 LightGCN trên cùng user, từng seed |
| Được cải thiện / bị giảm | Δ > 0 (hoặc < 0) ở cả 3 seeds; không đổi: Δ = 0 ở cả 3 seeds; còn lại: không nhất quán |
| Độ phổ biến sách test | Trung bình số tương tác train của các sách test của user |
| Mức khớp profile–sách test | cos(user emb, item emb) trung bình trên sách test − trung bình trên 50 sách ngẫu nhiên (dùng test, chỉ để giải thích) |
| Mức khớp profile–sách train | Như trên nhưng với sách train (chỉ dùng train, có thể dùng để điều chỉnh alignment) |

## CELL 1 — Libraries, Google Drive và đường dẫn

In [ ]:
import json, re, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import spearmanr
from IPython.display import display

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running on Google Colab; using local ROOT if configured.')

ROOT = Path('/content/drive/MyDrive/Data_SNA_Group14')
DATA_DIR = ROOT / 'data/raw/amazon'
PRED_DIR = ROOT / 'experiments/predictions'
TITLES_PATH = ROOT / 'data/manifests/amazon_book_titles.json'
OUT_DIR = ROOT / 'experiments/E3/error_analysis'
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEEDS = (2023, 2024, 2025)
K = 20
RNG_SEED = 0
EXPORTS = {m: {s: PRED_DIR / f'{m}/seed_{s}/topk_test.npz' for s in SEEDS} for m in ('lightgcn', 'lightgcn_plus')}
assert all(p.is_file() for runs in EXPORTS.values() for p in runs.values()), 'Missing Top-K exports'

## CELL 2 — Load dữ liệu, profiles và semantic embeddings

In [ ]:
def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

def binary_csr(m):
    m = m.tocsr(copy=True); m.sum_duplicates(); m.data = (m.data != 0).astype(np.int8); m.eliminate_zeros(); m.sort_indices()
    return m

train = binary_csr(load_pkl(DATA_DIR / 'trn_mat.pkl'))
test = binary_csr(load_pkl(DATA_DIR / 'tst_mat.pkl'))
usr_prf = load_pkl(DATA_DIR / 'usr_prf.pkl')
itm_prf = load_pkl(DATA_DIR / 'itm_prf.pkl')
ue = np.asarray(load_pkl(DATA_DIR / 'usr_emb_np.pkl'), dtype=np.float64)
ie = np.asarray(load_pkl(DATA_DIR / 'itm_emb_np.pkl'), dtype=np.float64)
ue /= np.linalg.norm(ue, axis=1, keepdims=True)
ie /= np.linalg.norm(ie, axis=1, keepdims=True)
titles = {r['iid']: (r['title'] or 'NO_TITLE') for r in json.load(open(TITLES_PATH))}
n_users, n_items = train.shape
degree = np.asarray(train.sum(axis=0)).ravel()
print(f'Users={n_users:,} Items={n_items:,} emb={ue.shape[1]}')

## CELL 3 — Recall@20 theo từng user, ΔRecall và phân nhóm user

In [ ]:
def per_user_recall(path):
    z = np.load(path, allow_pickle=False)
    r = np.full(n_users, np.nan)
    for u, row in zip(z['user_ids'], z['item_ids'][:, :K]):
        t = test.indices[test.indptr[u]:test.indptr[u + 1]]
        if len(t):
            r[u] = np.isin(row, t).sum() / len(t)
    return r

rec = {m: np.vstack([per_user_recall(EXPORTS[m][s]) for s in SEEDS]) for m in EXPORTS}
lg, rl = rec['lightgcn'], rec['lightgcn_plus']
delta = rl - lg
valid = ~np.isnan(delta[0])
improved = valid & (delta > 0).all(0)
worsened = valid & (delta < 0).all(0)
unchanged = valid & (delta == 0).all(0)
mixed = valid & ~improved & ~worsened & ~unchanged
both_zero = valid & (lg == 0).all(0) & (rl == 0).all(0)

n_train = np.diff(train.indptr)
def mean_deg(m, u):
    i = m.indices[m.indptr[u]:m.indptr[u + 1]]
    return degree[i].mean() if len(i) else np.nan
test_deg = np.array([mean_deg(test, u) for u in range(n_users)])

rng = np.random.default_rng(RNG_SEED)
def sem_gap(m, u):
    i = m.indices[m.indptr[u]:m.indptr[u + 1]]
    if not len(i):
        return np.nan
    rand = rng.choice(n_items, 50, replace=False)
    return (ie[i] @ ue[u]).mean() - (ie[rand] @ ue[u]).mean()
gap_test = np.array([sem_gap(test, u) for u in range(n_users)])
gap_train = np.array([sem_gap(train, u) for u in range(n_users)])

users = pd.DataFrame({'uid': np.arange(n_users), 'n_train': n_train, 'test_item_degree': test_deg,
                      'gap_test': gap_test, 'gap_train': gap_train,
                      'recall_lightgcn': lg.mean(0), 'recall_rlmrec': rl.mean(0), 'delta_mean': delta.mean(0)})
users['group'] = np.select([improved, worsened, unchanged, mixed], ['improved', 'worsened', 'unchanged', 'mixed'], 'no_test')
users.to_csv(OUT_DIR / 'per_user_delta.csv', index=False)

group_table = (users[valid].groupby('group')
               .agg(n_users=('uid', 'size'), mean_train_items=('n_train', 'mean'),
                    mean_test_item_degree=('test_item_degree', 'mean'),
                    recall_lightgcn=('recall_lightgcn', 'mean'), recall_rlmrec=('recall_rlmrec', 'mean'))
               .reindex(['improved', 'worsened', 'unchanged', 'mixed']))
group_table.to_csv(OUT_DIR / 'user_groups.csv')
print('Users with test:', valid.sum(), '| unchanged with zero recall in both models:', both_zero.sum())
display(group_table.round(3))

## CELL 4 — ΔRecall@20 theo 5 mức của từng đặc điểm
Mỗi mức gồm 20% user có test. Mức 1 = thấp nhất.

In [ ]:
rows = []
for feat, label in [('test_item_degree', 'Độ phổ biến TB của sách test'),
                    ('gap_test', 'Mức khớp profile–sách test'),
                    ('gap_train', 'Mức khớp profile–sách train (chỉ train)'),
                    ('n_train', 'Số sách train')]:
    d = users[valid & ~users[feat].isna()]
    bins = pd.qcut(d[feat].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
    means = d.groupby(bins, observed=True)['delta_mean'].mean()
    nz = d[~both_zero[d.uid]]
    rows.append({'feature': label, **{f'level_{k}': means[k] for k in range(1, 6)},
                 'spearman_users_with_hits': spearmanr(nz[feat], nz['delta_mean']).correlation})
levels = pd.DataFrame(rows)
levels.to_csv(OUT_DIR / 'delta_by_level.csv', index=False)
display(levels.round(4))

## CELL 5 — Phía item: sách được gợi ý đúng nhiều hơn / ít hơn

In [ ]:
def item_hits(paths):
    H, X = [], []
    for p in paths:
        z = np.load(p, allow_pickle=False)
        h = np.zeros(n_items); x = np.zeros(n_items)
        for u, row in zip(z['user_ids'], z['item_ids'][:, :K]):
            t = test.indices[test.indptr[u]:test.indptr[u + 1]]
            x[row] += 1
            h[row[np.isin(row, t)]] += 1
        H.append(h); X.append(x)
    return np.mean(H, 0), np.mean(X, 0)

hits_lg, exp_lg = item_hits(EXPORTS['lightgcn'].values())
hits_rl, exp_rl = item_hits(EXPORTS['lightgcn_plus'].values())
uninformative = re.compile(r'not provide enough|unable to|insufficient|no (description|information)|cannot determine|not enough information|^none\b|difficult to determine', re.I)
items = pd.DataFrame({'iid': np.arange(n_items), 'title': [titles[i] for i in range(n_items)], 'train_degree': degree,
                      'hits_lightgcn': hits_lg, 'hits_rlmrec': hits_rl, 'exposure_lightgcn': exp_lg, 'exposure_rlmrec': exp_rl,
                      'profile_uninformative': [bool(uninformative.search((itm_prf[i].get('profile') or '')[:400]))
                                                or len((itm_prf[i].get('profile') or '').split()) < 15 for i in range(n_items)]})
items['delta_hits'] = items.hits_rlmrec - items.hits_lightgcn
items.to_csv(OUT_DIR / 'per_item_delta.csv', index=False)
cols = ['iid', 'title', 'train_degree', 'hits_lightgcn', 'hits_rlmrec', 'exposure_lightgcn', 'exposure_rlmrec']
print('Most gained'); display(items.nlargest(5, 'delta_hits')[cols].round(1))
print('Most lost'); display(items.nsmallest(5, 'delta_hits')[cols].round(1))
print('Uninformative item profiles:', items.profile_uninformative.sum())

## CELL 6 — Case study
Chọn user có 6–14 sách train. Được cải thiện: xếp theo mức khớp profile–sách test giảm dần. Bị giảm: xếp theo độ phổ biến sách test giảm dần. Hiển thị Top-K của seed 2023.

In [ ]:
CASE_SEED = 2023
z_lg = np.load(EXPORTS['lightgcn'][CASE_SEED]); z_rl = np.load(EXPORTS['lightgcn_plus'][CASE_SEED])
row_of = {int(u): i for i, u in enumerate(z_lg['user_ids'])}

def show_user(u):
    t = set(test.indices[test.indptr[u]:test.indptr[u + 1]].tolist())
    a = z_lg['item_ids'][row_of[u]][:K]; b = z_rl['item_ids'][row_of[u]][:K]
    print(f'\n=== user {u} | train {n_train[u]} | test {len(t)} | Recall@20 LightGCN {np.round(lg[:, u], 2)} RLMRec {np.round(rl[:, u], 2)} | gap_test {gap_test[u]:.3f}')
    print('PROFILE:', usr_prf[u]['profile'][:400])
    print('TRAIN  :', [(titles[i][:40], int(degree[i])) for i in train.indices[train.indptr[u]:train.indptr[u + 1]]])
    print('TEST   :', [(titles[i][:40], int(degree[i]), 'LightGCN hit' if i in a else '', 'RLMRec hit' if i in b else '') for i in t])
    print('LightGCN top5:', [(titles[i][:32], int(degree[i])) for i in a[:5]])
    print('RLMRec   top5:', [(titles[i][:32], int(degree[i])) for i in b[:5]])

pool = users[(users.n_train.between(6, 14)) & users.uid.isin(row_of.keys())]
for u in pool[pool.group == 'improved'].nlargest(3, 'gap_test').uid:
    show_user(int(u))
print('\n######## WORSENED')
for u in pool[pool.group == 'worsened'].nlargest(3, 'test_item_degree').uid:
    show_user(int(u))